# Chapter 8. Data Wrangling: Join, Combine, and Reshape

## 8.1 Hierarchial Indexing
- Hierarchial indexing is an important feature of pandas that enables you to have multiple (two or more) index levels on an axis.

In [1]:
import pandas as pd
import numpy as np

In [10]:
data = pd.Series(np.random.randn(9),
                index=[['a','a','a','b','b','c','c','d','d'],
                      [1,2,3,1,3,1,2,2,3]])
data

a  1   -0.728143
   2    1.486698
   3    0.798631
b  1   -1.569140
   3   -0.970689
c  1   -0.593116
   2    0.189080
d  2   -0.713954
   3    0.408746
dtype: float64

> - 아래는 *Partial* indexing 예시 

In [11]:
data['b']

1   -1.569140
3   -0.970689
dtype: float64

In [12]:
data['b':'c']

b  1   -1.569140
   3   -0.970689
c  1   -0.593116
   2    0.189080
dtype: float64

In [13]:
data.loc[['b','d'],3]

b  3   -0.970689
d  3    0.408746
dtype: float64

In [14]:
data.loc[:, 2]

a    1.486698
c    0.189080
d   -0.713954
dtype: float64

> - You could rearrange the data into a **DataFrame** using its ***unstack*** method:

In [15]:
data.unstack() # data.unstack().stack()

,1,2,3
a,-0.728143,1.486698,0.798631
b,-1.569140,NaN,-0.970689
c,-0.593116,0.189080,NaN
d,NaN,-0.713954,0.408746


> - With a DataFrame, either axis can have a hierarchial index:

In [22]:
frame = pd.DataFrame(np.arange(12).reshape((4,3)),
                    index=[['a','a','b','b'], [1,2,1,2]],
                    columns=[['Ohio','Ohio','Colorado'], ['Green','Red','Green']])

# The hierarchial levels can have names
frame.index.names = ['key1','key2']
frame.columns.names = ['state', 'color']

frame

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
     2        3   4        5
b    1        6   7        8
     2        9  10       11

### Reordering and Sorting Levels

> - The ***swaplevel*** takes two level numbers or names and returns a new object with the levels interchanged (but the data is otherwise unaltered):

In [39]:
frame.swaplevel('key1', 'key2') # axis=1면 column name 기준으로 swap

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
2    a        3   4        5
1    b        6   7        8
2    b        9  10       11

> - ***sort_index***, on the other hand, sorts the data using only the values in a single level.

In [43]:
frame.sort_index(level=1) # = 'key2'로 sort

state      Ohio     Colorado
color     Green Red    Green
key1 key2                   
a    1        0   1        2
b    1        6   7        8
a    2        3   4        5
b    2        9  10       11

In [45]:
frame.swaplevel(0,1).sort_index(level=0)

state      Ohio     Colorado
color     Green Red    Green
key2 key1                   
1    a        0   1        2
     b        6   7        8
2    a        3   4        5
     b        9  10       11

### Summary Statistical by Level

In [48]:
frame.groupby(level='key2').sum()

state  Ohio     Colorado
color Green Red    Green
key2                    
1         6   8       10
2        12  14       16

In [50]:
frame.groupby(level='color', axis=1).sum()

color      Green  Red
key1 key2            
a    1         2    1
     2         8    4
b    1        14    7
     2        20   10

### Indexing with a DataFrame's columns

In [2]:
frame = pd.DataFrame({'a':range(7), 'b':range(7,0,-1),
                     'c':['one','one','one','two','two','two','two'],
                     'd':[0,1,2,0,1,2,3]})
frame

,a,b,c,d
0,0,7,one,0
1,1,6,one,1
2,2,5,one,2
3,3,4,two,0
4,4,3,two,1
5,5,2,two,2
6,6,1,two,3


> - DataFrame's ***set_index*** function will create a new DataFrame using one or more of its columns as the index:
> - ***reset_index***, on the other hand, does the opposite of *set_index*

In [5]:
frame2 = frame.set_index(['c','d']) # drop=False 조건 부여하면 DF 내에 데이터도 유지됨
frame2

a  b
c   d      
one 0  0  7
    1  1  6
    2  2  5
two 0  3  4
    1  4  3
    2  5  2
    3  6  1